# Solutions to Exercises on Bayesian Networks

For each situation we give one reasonable model. There is no single correct answer: different orderings and parent sets can be defended, and the probabilities are only plausible illustrative values. In each case the node ordering puts causes before effects. Arrays are indexed in the same order as their names, and index 1 means "True" (so `PA_CF[1][c][f]` is $P(A=1\vert C=c,F=f)$). The function `cpt` builds a full table from the probabilities $P(X=1\vert\text{parents})$.

In [1]:
import numpy as np

import html as _html
from pyvis.network import Network      # pip install pyvis
from IPython.display import HTML, display

def draw_dag(pos, edges, title=None, scale=110, cdn_resources='remote'):
    """Draw a DAG interactively with pyvis (the nodes can be dragged with the mouse).
    pos: {node: (x, y)} with y increasing upwards, edges: list of (parent, child).
    cdn_resources='remote' loads the vis.js library from the internet; use 'in_line' to embed it (offline use)."""
    xs, ys = zip(*pos.values())
    height = int((max(ys) - min(ys)) * scale + 140)
    net = Network(height=f"{height}px", width="100%", directed=True, notebook=True,
                  cdn_resources=cdn_resources)
    for n, (x, y) in pos.items():
        net.add_node(n, label=n, x=x*scale, y=-y*scale, physics=False, shape="circle",
                     color={"background": "#E6F1FB", "border": "#378ADD"},
                     font={"size": 20, "color": "#042C53"}, margin=12)
    for parent, child in edges:
        net.add_edge(parent, child, color="#555555", width=2, arrows="to")
    net.set_options('{"physics": {"enabled": false}, "edges": {"smooth": false}}')
    page = net.generate_html(notebook=True)
    heading = f"<p><b>{title}</b></p>" if title else ""
    display(HTML(f'<div>{heading}<iframe srcdoc="{_html.escape(page, quote=True)}" width="100%" '
                 f'height="{height + 40}" style="border:0"></iframe></div>'))

def cpt(p1):
    """Conditional probability table for a binary variable from the array p1 = P(X=1 | parents).
    Returns an array indexed [x][parents...] with [0] = P(X=0|...) and [1] = P(X=1|...)."""
    p1 = np.asarray(p1, dtype=float)
    return np.array([1 - p1, p1])

def posterior(PJ, names, query, evidence={}):
    """P(query | evidence) from a joint distribution PJ with one axis per variable in `names`.
    evidence is a dictionary {variable name: value}."""
    index = tuple(evidence.get(n, slice(None)) for n in names)
    sub = PJ[index]
    remaining = [n for n in names if n not in evidence]
    other_axes = tuple(i for i, n in enumerate(remaining) if n != query)
    p = sub.sum(axis=other_axes)
    return p / p.sum()

In [2]:
# Helper: count free parameters, given the list of parents of each (binary) node
def n_params(parents):
    return sum(2**len(p) for p in parents.values())

def report(parents):
    N = len(parents)
    print(f"Free parameters: {n_params(parents)}   (full joint distribution of {N} binary variables: {2**N - 1})")

## 1. Cooking dinner, house on fire, smoke alarm

Variables: Cooking $C$, Fire $F$, Alarm $A$. Ordering: $C,F,A$. Edges: $C\to F$ (cooking is a common cause of house fires), $C\to A$ and $F\to A$ (the alarm is triggered by smoke from a fire, or from cooking).

$$P(C,F,A)=P(C)P(F\vert C)P(A\vert C,F)$$

**Conditional probability tables.** Each table gives $P(X\vert\text{parents})$ for both values of $X$.

| $P(C)$ | $P(\lnot C)$ |
|:--:|:--:|
| 0.2 | 0.8 |

| | $P(F\vert C)$ | $P(\lnot F\vert C)$ |
|:--|:-:|:-:|
| $C$ | 0.001 | 0.999 |
| $\lnot C$ | 0.0001 | 0.9999 |

| | $P(A\vert C,F)$ | $P(\lnot A\vert C,F)$ |
|:--|:-:|:-:|
| $C,F$ | 0.99 | 0.01 |
| $C,\lnot F$ | 0.1 | 0.9 |
| $\lnot C,F$ | 0.99 | 0.01 |
| $\lnot C,\lnot F$ | 0.001 | 0.999 |

In [3]:
parents = {'C': [], 'F': ['C'], 'A': ['C', 'F']}
report(parents)

PC = cpt(0.2)
PF_C = cpt([0.0001, 0.001])                      # P(F | C), C = 0, 1
PA_CF = cpt([[0.001, 0.99], [0.10, 0.99]])       # P(A | C,F)[a][c][f]
for name, P in [('PC', PC), ('PF_C', PF_C), ('PA_CF', PA_CF)]:
    print(name, "normalised:", np.allclose(P.sum(axis=0), 1))

names = ['C', 'F', 'A']
PCFA = np.einsum('c,fc,acf->cfa', PC, PF_C, PA_CF)

print("P(F=1)            =", round(posterior(PCFA, names, 'F')[1], 4))
print("P(F=1 | A=1)      =", round(posterior(PCFA, names, 'F', {'A': 1})[1], 4))
print("P(F=1 | A=1, C=1) =", round(posterior(PCFA, names, 'F', {'A': 1, 'C': 1})[1], 4))
print("P(F=1 | A=1, C=0) =", round(posterior(PCFA, names, 'F', {'A': 1, 'C': 0})[1], 4))

Free parameters: 7   (full joint distribution of 3 binary variables: 7)
PC normalised: True
PF_C normalised: True
PA_CF normalised: True
P(F=1)            = 0.0003
P(F=1 | A=1)      = 0.0132
P(F=1 | A=1, C=1) = 0.0098
P(F=1 | A=1, C=0) = 0.0901


## 2. Heavy rain, burst water pipe, flooded patio, call plumber

Variables: Rain $R$, Burst pipe $B$, Flooded patio $P$, Call plumber $L$. Ordering: $R,B,P,L$. Edges: $R\to P$, $B\to P$ (the patio floods because of heavy rain or a burst pipe), $B\to L$, $P\to L$ (you call a plumber if you know the pipe has burst, or if you see flooding that you cannot explain). Rain and a burst pipe are modelled as independent. (There is a collider at $P$.)

$$P(R,B,P,L)=P(R)P(B)P(P\vert R,B)P(L\vert B,P)$$

**Conditional probability tables.** Each table gives $P(X\vert\text{parents})$ for both values of $X$.

| $P(R)$ | $P(\lnot R)$ |
|:--:|:--:|
| 0.15 | 0.85 |

| $P(B)$ | $P(\lnot B)$ |
|:--:|:--:|
| 0.01 | 0.99 |

| | $P(P\vert R,B)$ | $P(\lnot P\vert R,B)$ |
|:--|:-:|:-:|
| $R,B$ | 0.99 | 0.01 |
| $R,\lnot B$ | 0.3 | 0.7 |
| $\lnot R,B$ | 0.9 | 0.1 |
| $\lnot R,\lnot B$ | 0.001 | 0.999 |

| | $P(L\vert B,P)$ | $P(\lnot L\vert B,P)$ |
|:--|:-:|:-:|
| $B,P$ | 0.95 | 0.05 |
| $B,\lnot P$ | 0.6 | 0.4 |
| $\lnot B,P$ | 0.1 | 0.9 |
| $\lnot B,\lnot P$ | 0.001 | 0.999 |

In [4]:
parents = {'R': [], 'B': [], 'P': ['R', 'B'], 'L': ['B', 'P']}
report(parents)

PR = cpt(0.15)
PB = cpt(0.01)
PP_RB = cpt([[0.001, 0.90], [0.30, 0.99]])       # P(P | R,B)[p][r][b]
PL_BP = cpt([[0.001, 0.10], [0.60, 0.95]])       # P(L | B,P)[l][b][p]
for name, P in [('PR', PR), ('PB', PB), ('PP_RB', PP_RB), ('PL_BP', PL_BP)]:
    print(name, "normalised:", np.allclose(P.sum(axis=0), 1))

names = ['R', 'B', 'P', 'L']
PRBPL = np.einsum('r,b,prb,lbp->rbpl', PR, PB, PP_RB, PL_BP)

print("P(B=1)            =", round(posterior(PRBPL, names, 'B')[1], 4))
print("P(B=1 | P=1)      =", round(posterior(PRBPL, names, 'B', {'P': 1})[1], 4))
print("P(B=1 | P=1, R=1) =", round(posterior(PRBPL, names, 'B', {'P': 1, 'R': 1})[1], 4))
print("P(B=1 | L=1)      =", round(posterior(PRBPL, names, 'B', {'L': 1})[1], 4))

Free parameters: 10   (full joint distribution of 4 binary variables: 15)
PR normalised: True
PB normalised: True
PP_RB normalised: True
PL_BP normalised: True
P(B=1)            = 0.01
P(B=1 | P=1)      = 0.1675
P(B=1 | P=1, R=1) = 0.0323
P(B=1 | L=1)      = 0.6265


## 3. Late for train, speeding, speeding ticket, camera malfunction

Variables: Speeding $S$, Camera malfunction $M$, Ticket $T$, Late $L$. Ordering: $S,M,L,T$. Edges: $S\to L$ (speeding makes you less likely to be late), $S\to T$ and $M\to T$ (a ticket needs you to have been speeding *and* a camera that works). The camera is independent of whether you speed. The graph has a collider at $T$, which is exercise 7.

$$P(S,M,L,T)=P(S)P(M)P(L\vert S)P(T\vert S,M)$$

**Conditional probability tables.** Each table gives $P(X\vert\text{parents})$ for both values of $X$.

| $P(S)$ | $P(\lnot S)$ |
|:--:|:--:|
| 0.3 | 0.7 |

| $P(M)$ | $P(\lnot M)$ |
|:--:|:--:|
| 0.05 | 0.95 |

| | $P(L\vert S)$ | $P(\lnot L\vert S)$ |
|:--|:-:|:-:|
| $S$ | 0.15 | 0.85 |
| $\lnot S$ | 0.35 | 0.65 |

| | $P(T\vert S,M)$ | $P(\lnot T\vert S,M)$ |
|:--|:-:|:-:|
| $S,M$ | 0 | 1 |
| $S,\lnot M$ | 0.9 | 0.1 |
| $\lnot S,M$ | 0 | 1 |
| $\lnot S,\lnot M$ | 0.001 | 0.999 |

In [5]:
parents = {'S': [], 'M': [], 'L': ['S'], 'T': ['S', 'M']}
report(parents)

PS = cpt(0.3)
PM = cpt(0.05)
PL_S = cpt([0.35, 0.15])                         # P(L | S)
PT_SM = cpt([[0.001, 0.0], [0.9, 0.0]])          # P(T | S,M)[t][s][m]; no ticket if the camera has failed
for name, P in [('PS', PS), ('PM', PM), ('PL_S', PL_S), ('PT_SM', PT_SM)]:
    print(name, "normalised:", np.allclose(P.sum(axis=0), 1))

names = ['S', 'M', 'L', 'T']
PSMLT = np.einsum('s,m,ls,tsm->smlt', PS, PM, PL_S, PT_SM)

print("P(S=1)       =", round(posterior(PSMLT, names, 'S')[1], 4))
print("P(S=1 | L=1) =", round(posterior(PSMLT, names, 'S', {'L': 1})[1], 4))
print("P(S=1 | T=1) =", round(posterior(PSMLT, names, 'S', {'T': 1})[1], 4))

Free parameters: 8   (full joint distribution of 4 binary variables: 15)
PS normalised: True
PM normalised: True
PL_S normalised: True
PT_SM normalised: True
P(S=1)       = 0.3
P(S=1 | L=1) = 0.1552
P(S=1 | T=1) = 0.9974


## 4. Sprinkler, rain, wet grass

Variables: Rain $R$, Sprinkler $S$, Wet grass $W$. Ordering: $R,S,W$. Edges: $R\to S$ (people do not run the sprinkler when it is raining), $R\to W$, $S\to W$.

$$P(R,S,W)=P(R)P(S\vert R)P(W\vert R,S)$$

**Conditional probability tables.** Each table gives $P(X\vert\text{parents})$ for both values of $X$.

| $P(R)$ | $P(\lnot R)$ |
|:--:|:--:|
| 0.2 | 0.8 |

| | $P(S\vert R)$ | $P(\lnot S\vert R)$ |
|:--|:-:|:-:|
| $R$ | 0.01 | 0.99 |
| $\lnot R$ | 0.4 | 0.6 |

| | $P(W\vert R,S)$ | $P(\lnot W\vert R,S)$ |
|:--|:-:|:-:|
| $R,S$ | 0.99 | 0.01 |
| $R,\lnot S$ | 0.9 | 0.1 |
| $\lnot R,S$ | 0.9 | 0.1 |
| $\lnot R,\lnot S$ | 0.02 | 0.98 |

In [6]:
parents = {'R': [], 'S': ['R'], 'W': ['R', 'S']}
report(parents)

PR = cpt(0.2)
PS_R = cpt([0.4, 0.01])                          # P(S | R)
PW_RS = cpt([[0.02, 0.90], [0.90, 0.99]])       # P(W | R,S)[w][r][s]
for name, P in [('PR', PR), ('PS_R', PS_R), ('PW_RS', PW_RS)]:
    print(name, "normalised:", np.allclose(P.sum(axis=0), 1))

names = ['R', 'S', 'W']
PRSW = np.einsum('r,sr,wrs->rsw', PR, PS_R, PW_RS)

print("P(R=1)            =", round(posterior(PRSW, names, 'R')[1], 4))
print("P(R=1 | W=1)      =", round(posterior(PRSW, names, 'R', {'W': 1})[1], 4))
print("P(R=1 | W=1, S=1) =", round(posterior(PRSW, names, 'R', {'W': 1, 'S': 1})[1], 4))
print("P(R=1 | W=1, S=0) =", round(posterior(PRSW, names, 'R', {'W': 1, 'S': 0})[1], 4))

Free parameters: 7   (full joint distribution of 3 binary variables: 7)
PR normalised: True
PS_R normalised: True
PW_RS normalised: True
P(R=1)            = 0.2
P(R=1 | W=1)      = 0.3771
P(R=1 | W=1, S=1) = 0.0068
P(R=1 | W=1, S=0) = 0.9489


## 5. Car alarm sounding, earthquake, heavy traffic, attempted car theft, angry neighbours

Variables: Earthquake $E$, Attempted theft $T$, Heavy traffic $H$, Alarm $A$, Angry neighbours $N$. Ordering: $E,T,H,A,N$. Edges: $E\to A$, $T\to A$ (the alarm sounds if the car is shaken by an earthquake or someone tries to steal it), $A\to N$, $H\to N$ (neighbours are angry if the alarm is sounding, and are more irritable when traffic is heavy). The three root causes are modelled as independent.

$$P(E,T,H,A,N)=P(E)P(T)P(H)P(A\vert E,T)P(N\vert A,H)$$

**Conditional probability tables.** Each table gives $P(X\vert\text{parents})$ for both values of $X$.

| $P(E)$ | $P(\lnot E)$ |
|:--:|:--:|
| 0.001 | 0.999 |

| $P(T)$ | $P(\lnot T)$ |
|:--:|:--:|
| 0.01 | 0.99 |

| $P(H)$ | $P(\lnot H)$ |
|:--:|:--:|
| 0.3 | 0.7 |

| | $P(A\vert E,T)$ | $P(\lnot A\vert E,T)$ |
|:--|:-:|:-:|
| $E,T$ | 0.99 | 0.01 |
| $E,\lnot T$ | 0.9 | 0.1 |
| $\lnot E,T$ | 0.95 | 0.05 |
| $\lnot E,\lnot T$ | 0.001 | 0.999 |

| | $P(N\vert A,H)$ | $P(\lnot N\vert A,H)$ |
|:--|:-:|:-:|
| $A,H$ | 0.8 | 0.2 |
| $A,\lnot H$ | 0.6 | 0.4 |
| $\lnot A,H$ | 0.1 | 0.9 |
| $\lnot A,\lnot H$ | 0.02 | 0.98 |

In [7]:
parents = {'E': [], 'T': [], 'H': [], 'A': ['E', 'T'], 'N': ['A', 'H']}
report(parents)

PE = cpt(0.001)
PT = cpt(0.01)
PH = cpt(0.3)
PA_ET = cpt([[0.001, 0.95], [0.90, 0.99]])       # P(A | E,T)[a][e][t]
PN_AH = cpt([[0.02, 0.10], [0.60, 0.80]])        # P(N | A,H)[n][a][h]
for name, P in [('PE', PE), ('PT', PT), ('PH', PH), ('PA_ET', PA_ET), ('PN_AH', PN_AH)]:
    print(name, "normalised:", np.allclose(P.sum(axis=0), 1))

names = ['E', 'T', 'H', 'A', 'N']
PETHAN = np.einsum('e,t,h,aet,nah->ethan', PE, PT, PH, PA_ET, PN_AH)

print("P(T=1)            =", round(posterior(PETHAN, names, 'T')[1], 4))
print("P(T=1 | A=1)      =", round(posterior(PETHAN, names, 'T', {'A': 1})[1], 4))
print("P(T=1 | A=1, E=1) =", round(posterior(PETHAN, names, 'T', {'A': 1, 'E': 1})[1], 4))
print("P(T=1 | N=1)      =", round(posterior(PETHAN, names, 'T', {'N': 1})[1], 4))

Free parameters: 11   (full joint distribution of 5 binary variables: 31)
PE normalised: True
PT normalised: True
PH normalised: True
PA_ET normalised: True
PN_AH normalised: True
P(T=1)            = 0.01
P(T=1 | A=1)      = 0.8348
P(T=1 | A=1, E=1) = 0.011
P(T=1 | N=1)      = 0.1234


## 6. Puncture, fuel pump failure, recovery vehicle, large bill, late for work

Variables: Puncture $P$, Fuel pump failure $F$, Recovery vehicle $R$, Large bill $B$, Late $L$. Ordering: $P,F,R,B,L$. Edges: $P\to R$, $F\to R$ (a recovery vehicle is called out if you have a breakdown; a puncture can often be fixed by the driver), $R\to B$ and $F\to B$ (a recovery vehicle costs money, and a new fuel pump is expensive), $R\to L$ and $P\to L$ (waiting for recovery makes you late, but so does changing a wheel yourself).

$$P(P,F,R,B,L)=P(P)P(F)P(R\vert P,F)P(B\vert R,F)P(L\vert R,P)$$

**Conditional probability tables.** Each table gives $P(X\vert\text{parents})$ for both values of $X$.

| $P(P)$ | $P(\lnot P)$ |
|:--:|:--:|
| 0.05 | 0.95 |

| $P(F)$ | $P(\lnot F)$ |
|:--:|:--:|
| 0.01 | 0.99 |

| | $P(R\vert P,F)$ | $P(\lnot R\vert P,F)$ |
|:--|:-:|:-:|
| $P,F$ | 1 | 0 |
| $P,\lnot F$ | 0.3 | 0.7 |
| $\lnot P,F$ | 1 | 0 |
| $\lnot P,\lnot F$ | 0 | 1 |

| | $P(B\vert R,F)$ | $P(\lnot B\vert R,F)$ |
|:--|:-:|:-:|
| $R,F$ | 0.95 | 0.05 |
| $R,\lnot F$ | 0.3 | 0.7 |
| $\lnot R,F$ | 0.8 | 0.2 |
| $\lnot R,\lnot F$ | 0.02 | 0.98 |

| | $P(L\vert R,P)$ | $P(\lnot L\vert R,P)$ |
|:--|:-:|:-:|
| $R,P$ | 0.95 | 0.05 |
| $R,\lnot P$ | 0.9 | 0.1 |
| $\lnot R,P$ | 0.6 | 0.4 |
| $\lnot R,\lnot P$ | 0.05 | 0.95 |

In [8]:
parents = {'P': [], 'F': [], 'R': ['P', 'F'], 'B': ['R', 'F'], 'L': ['R', 'P']}
report(parents)

PP = cpt(0.05)
PF = cpt(0.01)
PR_PF = cpt([[0.0, 1.0], [0.3, 1.0]])            # P(R | P,F)[r][p][f]
PB_RF = cpt([[0.02, 0.80], [0.30, 0.95]])        # P(B | R,F)[b][r][f]
PL_RP = cpt([[0.05, 0.60], [0.90, 0.95]])        # P(L | R,P)[l][r][p]
for name, P in [('PP', PP), ('PF', PF), ('PR_PF', PR_PF), ('PB_RF', PB_RF), ('PL_RP', PL_RP)]:
    print(name, "normalised:", np.allclose(P.sum(axis=0), 1))

names = ['P', 'F', 'R', 'B', 'L']
PPFRBL = np.einsum('p,f,rpf,brf,lrp->pfrbl', PP, PF, PR_PF, PB_RF, PL_RP)

print("P(F=1)            =", round(posterior(PPFRBL, names, 'F')[1], 4))
print("P(F=1 | R=1)      =", round(posterior(PPFRBL, names, 'F', {'R': 1})[1], 4))
print("P(F=1 | R=1, P=1) =", round(posterior(PPFRBL, names, 'F', {'R': 1, 'P': 1})[1], 4))
print("P(F=1 | B=1)      =", round(posterior(PPFRBL, names, 'F', {'B': 1})[1], 4))

Free parameters: 14   (full joint distribution of 5 binary variables: 31)
PP normalised: True
PF normalised: True
PR_PF normalised: True
PB_RF normalised: True
PL_RP normalised: True
P(F=1)            = 0.01
P(F=1 | R=1)      = 0.4024
P(F=1 | R=1, P=1) = 0.0326
P(F=1 | B=1)      = 0.2839


## The graphs

In [9]:
graphs = [
 ('1. Cooking, fire, alarm', {'C': (0, 0), 'F': (1, 1), 'A': (2, 0)}, [('C','F'), ('C','A'), ('F','A')]),
 ('2. Rain, pipe, patio, plumber', {'R': (0, 2), 'B': (2, 2), 'P': (1, 1), 'L': (2, 0)}, [('R','P'), ('B','P'), ('B','L'), ('P','L')]),
 ('3. Speeding, camera, ticket, late', {'S': (0, 2), 'M': (2, 2), 'L': (0, 0), 'T': (2, 0)}, [('S','L'), ('S','T'), ('M','T')]),
 ('4. Rain, sprinkler, wet grass', {'R': (1, 2), 'S': (0, 1), 'W': (1, 0)}, [('R','S'), ('R','W'), ('S','W')]),
 ('5. Earthquake, theft, traffic, alarm, neighbours', {'E': (0, 2), 'T': (2, 2), 'A': (1, 1), 'H': (4, 2), 'N': (3, 0)}, [('E','A'), ('T','A'), ('A','N'), ('H','N')]),
 ('6. Puncture, pump, recovery, bill, late', {'P': (0, 2), 'F': (2, 2), 'R': (1, 1), 'B': (2, 0), 'L': (0, 0)}, [('P','R'), ('F','R'), ('R','B'), ('F','B'), ('R','L'), ('P','L')]),
]
for title, pos, edges in graphs:
    draw_dag(pos, edges, title=title)

## 7. The collider in situation 3

Ticket $T$ has two parents, speeding $S$ and camera malfunction $M$, which are not connected: $S\to T\leftarrow M$ is a collider. Speeding and camera malfunction are independent (they are roots), but become dependent once the ticket has been observed. Intuitively: if you were speeding and did *not* get a ticket, the camera is much more likely to have failed.

In [10]:
# Marginally independent
PSM = PSMLT.sum(axis=(2, 3))
print("S and M independent?          ", np.allclose(PSM, np.outer(PSM.sum(axis=1), PSM.sum(axis=0))))

# Given T = 0 (no ticket)
PSM_T0 = PSMLT.sum(axis=2)[:, :, 0]
PSM_T0 = PSM_T0 / PSM_T0.sum()
print("S and M independent given T=0?", np.allclose(PSM_T0, np.outer(PSM_T0.sum(axis=1), PSM_T0.sum(axis=0))))

names = ['S', 'M', 'L', 'T']
print("P(M=1)            =", round(posterior(PSMLT, names, 'M')[1], 4))
print("P(M=1 | T=0)      =", round(posterior(PSMLT, names, 'M', {'T': 0})[1], 4))
print("P(M=1 | T=0, S=1) =", round(posterior(PSMLT, names, 'M', {'T': 0, 'S': 1})[1], 4))
print("P(M=1 | T=0, S=0) =", round(posterior(PSMLT, names, 'M', {'T': 0, 'S': 0})[1], 4))

S and M independent?           True
S and M independent given T=0? False
P(M=1)            = 0.05
P(M=1 | T=0)      = 0.0673
P(M=1 | T=0, S=1) = 0.3448
P(M=1 | T=0, S=0) = 0.05


## 8. Two networks for sprinkler, rain and wet grass

The version with $R\to S$ is more plausible, because people really do avoid watering the garden when it is raining: the sprinkler and the rain are not independent. The version with no edge between $R$ and $S$ assumes that they are independent, which is a simplification that may be acceptable if, for example, the sprinkler runs on a timer that ignores the weather. The first network needs $1+2+4=7$ parameters ($P(R)$, $P(S\vert R)$ and $P(W\vert R,S)$) and the second needs $1+1+4=6$ ($P(R)$, $P(S)$ and $P(W\vert R,S)$). The first has no saving at all compared with the $2^3-1=7$ of the full joint distribution, since every variable depends on all of its predecessors, and the second saves only one parameter. The saving from dropping edges becomes large only for larger networks.